# 06 · DeBERTa (eksploratif)
Dua sumber: `deberta_base` (embedding mean-pooling → PCA) dan `deberta_ft` (probabilitas sentimen fine-tuned). Protokol sama dengan 03 (tuning di Train, expanding CV + gap + 1-SE). A dan B diambil dari cache 03. Tidak ada seleksi memakai Test, jadi Val dan Test dilaporkan untuk semua sel.

In [1]:
import sys; sys.path.insert(0, "../src")
import json, joblib
import numpy as np, pandas as pd
import t3
from t3 import CFG, OUT, PRICE, SEED
from eval_utils import *

X = pd.read_parquet(OUT / "features.parquet")
blocks = json.loads((OUT / "blocks.json").read_text())
prices, articles, sent = t3.load_export()
days = sent.index
R = joblib.load(OUT / "tuned.joblib")          # hasil 03 (A, B, dan model harga untuk late fusion)

# fitur harian DeBERTa, sejajar dengan baris X
Eb, Ab = t3.load_emb("deberta_base", articles)
emb = t3.daily_emb(articles, Eb, days).reindex(prices.date).set_axis(prices.index).add_prefix("emb_deberta_base_")
P = np.load(t3.EXPORT / "deberta_ft_probs.npy"); meta = json.loads((t3.EXPORT / "deberta_ft_probs.json").read_text())
t3._check(meta, articles, len(P))
short = {"bullish": "pos", "positive": "pos", "bearish": "neg", "negative": "neg", "neutral": "neu"}
P = pd.DataFrame(P, columns=[short[l] for l in meta["labels"]])
ft = t3.daily_finbert(articles, P, days).reindex(prices.date).set_axis(prices.index)[["fb_mean", "fb_pos", "fb_neg", "fb_neu"]].add_prefix("dft_")
X = X.join(emb).join(ft)
TEXT = {"deberta_base": list(emb.columns), "deberta_ft": list(ft.columns)}
tr, va, te = (X[X.split == s] for s in ("train", "val", "test"))

## Tuning sel C (4 model × 2 sumber × early/late)

In [2]:
import time
CACHE = OUT / "tuned_deberta.joblib"
RD = joblib.load(CACHE) if CACHE.exists() else {}
for m in t3.MODELS:
    t0 = time.time()
    for s, cols in TEXT.items():
        for f in ("early", "late"):
            if (m, s, f) in RD: continue
            kw = dict(late_price=R[(m, "A", "early")]["models"][0]) if f == "late" else {}
            RD[(m, s, f)] = t3.tune(m, cols if f == "late" else PRICE + cols, tr, tr.target, **kw)
            joblib.dump(RD, CACHE)
    print(m, f"{time.time() - t0:.0f}s")

Ridge 19s
ElasticNet 12s
SVR 15s
GradientBoosting 32s


## Hasil (Val dan Test)

In [3]:
def row(key, models, D):
    p = t3.predict(models, D); e = evaluate(D.target, p)
    return {**{k: e[k] for k in ["RMSE", "MAE", "DirAcc", "PredStd"]}, "collapse": is_collapsed(p, D.target)}

cells = {**{(m, s, f): R[(m, s, f)] for m in t3.MODELS for s, f in (("A", "early"), ("B", "early"))}, **RD}
rows = []
for (m, s, f), r in cells.items():
    for name, D in (("Val", va), ("Test", te)):
        rows.append({"model": m, "set": s, "fusion": f, "split": name, **row((m, s, f), r["models"], D), "cv_rmse": r["cv_mse"] ** .5})
res = pd.DataFrame(rows); res.to_csv(OUT / "deberta_results.csv", index=False)
for name, D in (("Val", va), ("Test", te)):
    print(name, "| Prediksi 0 RMSE:", round(evaluate(D.target, np.zeros(len(D)))["RMSE"], 6), "| DirAcc mayoritas:", round(majority_baselines(D.target)["Mayoritas"], 3))
res.pivot_table(index=["model", "set", "fusion"], columns="split", values=["RMSE", "DirAcc"], sort=False).round(6)

Val | Prediksi 0 RMSE: 0.003382 | DirAcc mayoritas: 0.534
Test | Prediksi 0 RMSE: 0.002908 | DirAcc mayoritas: 0.591


RMSE              DirAcc          
split                                      Val      Test       Val      Test
model            set          fusion                                        
Ridge            A            early   0.003379  0.002893  0.534091  0.590909
                 B            early   0.003379  0.002893  0.534091  0.590909
                 deberta_base early   0.003369  0.002878  0.528409  0.568182
                              late    0.003384  0.002891  0.494318  0.596591
                 deberta_ft   early   0.003379  0.002893  0.534091  0.590909
                              late    0.003379  0.002893  0.534091  0.590909
ElasticNet       A            early   0.003373  0.002889  0.545455  0.590909
                 B            early   0.003373  0.002889  0.545455  0.590909
                 deberta_base early   0.003379  0.002894  0.534091  0.590909
                              late    0.003373  0.002889  0.539773  0.590909
                 deberta_ft   early   0.003373  0.002889  0.545455  0.590909
                              late    0.003373  0.002889  0.539773  0.590909
SVR              A            early   0.003381  0.002889  0.534091  0.596591
                 B            early   0.003395  0.002883  0.534091  0.590909
                 deberta_base early   0.003399  0.002898  0.539773  0.596591
                              late    0.003387  0.002903  0.551136  0.602273
                 deberta_ft   early   0.003388  0.002886  0.534091  0.590909
                              late    0.003384  0.002885  0.534091  0.596591
GradientBoosting A            early   0.003362  0.002956  0.551136  0.556818
                 B            early   0.003407  0.002947  0.528409  0.556818
                 deberta_base early   0.003376  0.002903  0.528409  0.573864
                              late    0.003391  0.002958  0.528409  0.568182
                 deberta_ft   early   0.003367  0.002927  0.534091  0.590909
                              late    0.003362  0.002923  0.545455  0.562500

## Signifikansi di Test (DM < 0: sel DeBERTa lebih akurat) dan kontrol negatif

In [4]:
y, z, sig = te.target, np.zeros(len(te)), []
for (m, s, f), r in RD.items():
    p = t3.predict(r["models"], te)
    sig.append({"model": m, "set": s, "fusion": f, "DM vs A": dm_test(y, p, t3.predict(R[(m, "A", "early")]["models"], te))[1],
                "DM vs Prediksi 0": dm_test(y, p, z)[1]})
display(pd.DataFrame(sig).rename(columns={"DM vs A": "p vs A", "DM vs Prediksi 0": "p vs Prediksi 0"}).round(3))

# kontrol negatif: sel DeBERTa terbaik di Val (tidak collapse), teksnya diacak
v = res[(res.split == "Val") & res.set.isin(TEXT) & ~res.collapse].sort_values("RMSE").iloc[0]
m, s, f = v.model, v.set, v.fusion
perm = np.random.default_rng(SEED).permutation(len(X))
Xc = X.copy(); Xc[TEXT[s]] = X[TEXT[s]].to_numpy()[perm]
trc, vac, tec = (Xc[Xc.split == k] for k in ("train", "val", "test"))
kw = dict(late_price=R[(m, "A", "early")]["models"][0]) if f == "late" else {}
ctrl = t3.tune(m, TEXT[s] if f == "late" else PRICE + TEXT[s], trc, trc.target, **kw)
print(f"Kontrol negatif untuk {m}/{s}/{f}")
pd.DataFrame({n: {sp: row(None, r["models"], D)["RMSE"] for sp, D in (("Val", Dv), ("Test", Dt))}
              for n, r, Dv, Dt in (("asli", RD[(m, s, f)], va, te), ("diacak", ctrl, vac, tec))}).round(6)

,model,set,fusion,p vs A,p vs Prediksi 0
0,Ridge,deberta_base,early,0.577,0.333
1,Ridge,deberta_base,late,0.867,0.444
2,Ridge,deberta_ft,early,0.100,0.231
3,Ridge,deberta_ft,late,0.378,0.240
4,ElasticNet,deberta_base,early,0.432,0.250
5,ElasticNet,deberta_base,late,0.456,0.154
6,ElasticNet,deberta_ft,early,NaN,0.140
7,ElasticNet,deberta_ft,late,0.456,0.154
8,SVR,deberta_base,early,0.663,0.789
9,SVR,deberta_base,late,0.576,0.925


Kontrol negatif untuk GradientBoosting/deberta_ft/late


,asli,diacak
Val,0.003362,0.003465
Test,0.002923,0.002996


## MLP (tanpa tuning, hyperparameter tetap sama dengan 05, 5 seed, early fusion)

In [5]:
from sklearn.neural_network import MLPRegressor
mlp = lambda seed: MLPRegressor(hidden_layer_sizes=(32, 16), alpha=1e-2, max_iter=300, random_state=seed)
SETS = {"A": PRICE, "B": PRICE + blocks["sent_lama"], **{s: PRICE + c for s, c in TEXT.items()}}
rows = []
for s, cols in SETS.items():
    models = [t3.make_pipe(cols, mlp(sd)).fit(tr, tr.target) for sd in range(SEED, SEED + 5)]
    for name, D in (("Val", va), ("Test", te)):
        ev = pd.DataFrame([evaluate(D.target, m.predict(D)) for m in models])
        rows.append({"set": s, "split": name, **ev.mean()[["MAE", "RMSE", "DirAcc", "PredStd"]], "RMSE_std": ev.RMSE.std(ddof=0),
                     "collapse": is_collapsed(t3.predict(models, D), D.target)})
mres = pd.DataFrame(rows); mres.to_csv(OUT / "deberta_mlp_results.csv", index=False)
for name, D in (("Val", va), ("Test", te)):
    print(name, "| Prediksi 0 RMSE:", round(evaluate(D.target, np.zeros(len(D)))["RMSE"], 6), "| DirAcc mayoritas:", round(majority_baselines(D.target)["Mayoritas"], 3))
mres.round(6)

Val | Prediksi 0 RMSE: 0.003382 | DirAcc mayoritas: 0.534
Test | Prediksi 0 RMSE: 0.002908 | DirAcc mayoritas: 0.591


,set,split,MAE,RMSE,DirAcc,PredStd,RMSE_std,collapse
0,A,Val,0.002729,0.003624,0.513636,0.001678,0.000093,False
1,A,Test,0.002511,0.003230,0.486364,0.001431,0.000084,False
2,B,Val,0.003194,0.004243,0.488636,0.002699,0.000121,False
3,B,Test,0.002900,0.003774,0.505682,0.002421,0.000146,False
4,deberta_base,Val,0.003000,0.003984,0.532955,0.002420,0.000132,False
5,deberta_base,Test,0.002785,0.003585,0.520455,0.002153,0.000104,False
6,deberta_ft,Val,0.002859,0.003769,0.500000,0.001891,0.000064,False
7,deberta_ft,Test,0.002471,0.003216,0.520455,0.001590,0.000067,False
